# Replenishment receipt forecast: data check
Upload `receipts_and_items_reviewed.xlsx` and `receipt_items_category_reviewed_with_quantity.csv` together in Google Colab, then run all cells. This notebook checks whether the reviewed receipts can support a next-month category forecast. A receipt line is not an independent month of history.

In [ ]:
from pathlib import Path
from io import BytesIO
from zipfile import ZipFile
import pandas as pd

reviewed = Path('receipts_and_items_reviewed.xlsx')
archive = Path('drive-download-20261026T233425Z-1-001.zip')
if not reviewed.exists() and not archive.exists():
    from google.colab import files
    files.upload()

if reviewed.exists():
    workbook = reviewed
elif archive.exists():
    with ZipFile(archive) as outer:
        if 'receipt_and_item_categorize.zip' not in outer.namelist():
            raise ValueError('Archive must contain receipt_and_item_categorize.zip')
        with ZipFile(BytesIO(outer.read('receipt_and_item_categorize.zip'))) as inner:
            workbook = BytesIO(inner.read('receipt_and_item_categorize/receipts_and_items.xlsx'))
else:
    raise FileNotFoundError('Upload the reviewed workbook or Drive ZIP')
sheets = pd.read_excel(workbook, sheet_name=['receipts', 'receipt_items'], dtype={'receipt_id': str})
receipts = sheets['receipts']
items = sheets['receipt_items']
required_receipts = {'receipt_id', 'purchase_date', 'transaction_type'}
required_items = {'receipt_id', 'line_number', 'category_name', 'review_status'}
if required_receipts - set(receipts) or required_items - set(items):
    raise ValueError('Required receipt or item columns are missing')
if 'review_status' not in receipts:
    receipts['review_status'] = 'unreviewed'
if receipts.receipt_id.isna().any() or receipts.receipt_id.duplicated().any():
    raise ValueError('Receipt IDs must be present and unique')
if items[['receipt_id', 'line_number']].isna().any().any() or items.duplicated(['receipt_id', 'line_number']).any():
    raise ValueError('Item receipt IDs and line numbers must be present and unique')
if not set(items.receipt_id).issubset(set(receipts.receipt_id)):
    raise ValueError('Some items have no matching receipt')
receipts['purchase_date'] = pd.to_datetime(receipts['purchase_date'], errors='raise')
category_file = Path('receipt_items_category_reviewed_with_quantity.csv')
pilot_items = pd.read_csv(category_file, dtype={'receipt_id': str}) if category_file.exists() else items
if category_file.exists():
    required_categories = {'receipt_id', 'line_number', 'quantity', 'category_name', 'category_review_status'}
    if required_categories - set(pilot_items) or pilot_items.duplicated(['receipt_id', 'line_number']).any() or not pilot_items.category_review_status.eq('approved_for_category_only').all():
        raise ValueError('Category review CSV has missing columns, duplicate lines, or unapproved categories')
print(f'{len(receipts)} receipts, {len(items)} item rows')

In [ ]:
purchases = receipts.loc[receipts.transaction_type.eq('purchase')].copy()
purchases['month'] = purchases.purchase_date.dt.to_period('M').astype(str)
print('Receipt types:', receipts.transaction_type.value_counts(dropna=False).to_dict())
print('Receipt review:', receipts.review_status.value_counts(dropna=False).to_dict())
print('Item review:', items.review_status.value_counts(dropna=False).to_dict())
if category_file.exists():
    print('Category-only review:', pilot_items.category_review_status.value_counts(dropna=False).to_dict())
print('Purchase months with confirmed dates:', purchases.loc[purchases.purchase_date.notna()].groupby('month').receipt_id.nunique().to_dict())
print('Purchases without confirmed dates:', int(purchases.purchase_date.isna().sum()))
print('Unclear categories:', int(items.category_name.fillna('').str.lower().eq('unsure').sum()))
item_approved = items.assign(ok=items.review_status.eq('approved') & items.category_name.notna() & ~items.category_name.str.lower().eq('unsure')).groupby('receipt_id').ok.all()
approved = purchases.loc[purchases.review_status.eq('approved') & purchases.purchase_date.notna() & purchases.receipt_id.map(item_approved).fillna(False)]
print('Approved purchase receipts with all items approved:', len(approved))
print('Approved purchase months:', approved.month.nunique())
if approved.month.nunique() < 3:
    print('STOP: fewer than three reviewed months. No validated forecast score is available; the pilot below is provisional.')
else:
    print('Next check: confirm which entire months have complete receipt coverage before building labels or testing a model.')

## Experimental receipt-category pilot
The next cell fits a simple category-frequency model on dated September purchases, then compares its top five categories with categories observed on receipts dated through October 13. It uses the category-only review CSV when uploaded. This is a provisional check, not validated monthly accuracy: neither month has confirmed complete coverage, 11 unknown products are excluded, and October was still in progress. It then prints a private November trial list from all dated purchases. Do not show it as a live Store forecast.

In [ ]:
from collections import Counter

dated = purchases.loc[purchases.purchase_date.notna(), ['receipt_id', 'purchase_date', 'month']]
events = pilot_items.loc[pilot_items.category_name.notna() & ~pilot_items.category_name.str.lower().eq('unsure'), ['receipt_id', 'category_name']].merge(dated, on='receipt_id').drop_duplicates(['receipt_id', 'category_name'])
september = events.loc[events.month.eq('2026-09')]
october = events.loc[events.month.eq('2026-10')]
assert september.purchase_date.max() < october.purchase_date.min(), 'Training must precede the observed test receipts'
def top_five(rows):
    counts = Counter(rows.category_name)
    return [name for name, _ in sorted(counts.items(), key=lambda entry: (-entry[1], entry[0]))[:5]]
october_prediction = top_five(september)
october_observed = set(october.category_name)
correct = len(set(october_prediction) & october_observed)
precision = correct / len(october_prediction)
recall = correct / len(october_observed)
f1 = 2 * precision * recall / (precision + recall) if precision + recall else 0
print('PROVISIONAL September-trained top five:', october_prediction)
print('Categories seen in October receipts through October 13:', sorted(october_observed))
print(f'Provisional precision {precision:.3f}, recall {recall:.3f}, F1 {f1:.3f}; one incomplete test month')
print('PRIVATE November baseline trial from all dated receipts:', top_five(events))

## What must happen next
Correct and approve receipt and item rows, resolve `unsure` categories, and record which months have complete receipt coverage. Three reviewed months are only an initial data gate, not proof of enough history. Compare the frequency baseline with a candidate model only when enough complete later months exist. November 2026 can be scored only after that month's receipts are complete.